### Timing: Depending size of the image, 10 min - 1 h

The registered coordinates were used to generate a grid of square polygons, the size of which was determined by the experimental settings. We first intersected these square polygons with the cell segmentation polygons, which may have been pre-annotated based on transcriptomic data. The area of the square polygon was termed the "ablation area". The overlapping region between a square polygon and a cell segmentation polygon was defined as the "sampling area". The sampling specificity was calculated as the ratio of the sampling area to the total intersection area within the ablation area. The sampling proportion was calculated as the ratio of the total intersection area of the sampling area to the ablation area. Only intersections meeting Criterion 1 (sampling area > 0.1 × ablation area) were considered for the subsequent criteria. Subsequently, only cells meeting Criterion 2 (sampling proportion > 0.30 and sampling specificity > 0.80) were retained for downstream analysis. If two cells from the same classification shared an ablation square, the sampling specificity criterion was ignored and the value was set to 1.0. For a single cell, the total sampling area was recorded as the sum of the products of the sampling area and its corresponding sampling specificity for each square polygon. Finally, for a given metabolite within a single square polygon, the raw intensity was adjusted based on the sampling area and sampling specificity.

environment:   OpenFISH_decode
This notebook is used to transfer the ion signals into single cells.

In [1]:
import scanpy as sc
from shapely import Polygon
import geopandas as gpd
import numpy as np
from spatialdata import read_zarr
import pandas as pd
from scipy.sparse import csr_matrix

/home/duan/miniconda3/envs/sopa/lib/python3.10/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/home/duan/miniconda3/envs/sopa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [2]:
# Read into registered MALDI AnnData
adata_m = sc.read_h5ad("demo_data/DHB_A3_uMAIA_aligned.h5ad")
adata_m

/home/duan/miniconda3/envs/sopa/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 34500 × 1117
    obs: 'x_raw', 'y_raw', 'x_scaled', 'y_scaled', 'x_scaled_aligned', 'y_scaled_aligned'
    var: 'm/z'
    uns: 'img_shape'
    obsm: 'spatial', 'spatial_aligned'

In [3]:
# Generate GeoDataframe with MALDI sampling squares polygons as geometry
polygon_list = []

for _,row in adata_m.obs.iterrows():
    
    TopLeft_x = row['x_scaled_aligned']
    TopLeft_y = row['y_scaled_aligned']
    
    arr = np.array([[TopLeft_x, TopLeft_y], [TopLeft_x + 20, TopLeft_y], [TopLeft_x + 20, TopLeft_y + 20], [TopLeft_x, TopLeft_y + 20], [TopLeft_x, TopLeft_y]])
    polygon_list.append(Polygon(arr))

MALDI_gpd = gpd.GeoDataFrame(geometry=polygon_list)
MALDI_gpd.index = MALDI_gpd.index.astype(str)
MALDI_gpd

,geometry
0,"POLYGON ((5296.033 2.752, 5316.033 2.752, 5316..."
1,"POLYGON ((5357.321 3.335, 5377.321 3.335, 5377..."
2,"POLYGON ((5418.609 3.917, 5438.609 3.917, 5438..."
3,"POLYGON ((5479.897 4.5, 5499.897 4.5, 5499.897..."
4,"POLYGON ((5541.185 5.083, 5561.185 5.083, 5561..."
...,...
34495,"POLYGON ((16399.77 11688.116, 16419.77 11688.1..."
34496,"POLYGON ((16461.31 11688.379, 16481.31 11688.3..."
34497,"POLYGON ((16522.849 11688.643, 16542.849 11688..."
34498,"POLYGON ((16584.389 11688.906, 16604.389 11688..."


In [4]:
# We only provide two cycles in demo data for decoding.
# So, to enable cell-type-aware signal transfer, we provide pre-annotated dataset.
adata = sc.read_h5ad("demo_data/DHB_A3_OpenFISH_annotated.h5ad")
adata = adata[adata.obs['Sample'] == 'DHB_5M_20um_8um_A3'].copy()
adata

AnnData object with n_obs × n_vars = 22028 × 99
    obs: 'region', 'Name', 'x', 'y', 'area', 'leiden', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_FP', 'log1p_total_counts_FP', 'pct_counts_FP', 'log1p_area', 'Sample', 'cell_type', 'Batch'
    uns: 'Batch_colors', 'Sample_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap', 'spatial'
    varm: 'PCs'
    layers: 'counts', 'counts_corrected'
    obsp: 'connectivities', 'distances'

In [6]:
# Read into pre-decoded whole spatialdata.
sdata = read_zarr("demo_data/A3_raw_sdata.zarr/")

version mismatch: detected: RasterFormatV02, requested: FormatV04
/home/duan/miniconda3/envs/sopa/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


In [7]:
st_polygon = sdata.shapes['cell_boundaries'].copy()
st_polygon.index = st_polygon.index.astype(str)
st_polygon

,geometry
0,"POLYGON ((2489.97 11767.215, 2489.012 11767.29..."
1,"POLYGON ((2643.367 8412.213, 2642.967 8410.188..."
2,"POLYGON ((7978.592 5059.221, 7977.589 5059.267..."
3,"POLYGON ((6663.322 942.762, 6662.893 939.591, ..."
4,"POLYGON ((4819.988 2127.38, 4819.819 2124.872,..."
...,...
51002,"POLYGON ((13410.131 18567.245, 13409.934 18567..."
51003,"POLYGON ((13368.576 18577.918, 13368.533 18578..."
51004,"POLYGON ((13397.867 18570.66, 13397.676 18570...."
51005,"POLYGON ((10195.299 18581.947, 10195.24 18582...."


In [8]:
OpenFISH_gpd = st_polygon.loc[adata.obs_names, :].copy()
OpenFISH_gpd

,geometry
1,"POLYGON ((2643.367 8412.213, 2642.967 8410.188..."
2,"POLYGON ((7978.592 5059.221, 7977.589 5059.267..."
3,"POLYGON ((6663.322 942.762, 6662.893 939.591, ..."
4,"POLYGON ((4819.988 2127.38, 4819.819 2124.872,..."
5,"POLYGON ((4353.521 10250.902, 4346.733 10252.3..."
...,...
30481,"POLYGON ((15913.082 11686.5, 15906.44 11686.54..."
30488,"POLYGON ((15619.094 11690.751, 15619.073 11690..."
30495,"POLYGON ((16189.295 11692.782, 16151.671 11698..."
30635,"POLYGON ((15941.119 11731.139, 15897.656 11739..."


In [9]:
intersection = gpd.sjoin(OpenFISH_gpd, MALDI_gpd, how='inner', predicate='intersects', lsuffix='st', rsuffix='sm')
intersection

,geometry,index_sm
1,"POLYGON ((2643.367 8412.213, 2642.967 8410.188...",24815
2,"POLYGON ((7978.592 5059.221, 7977.589 5059.267...",12252
2,"POLYGON ((7978.592 5059.221, 7977.589 5059.267...",12458
3,"POLYGON ((6663.322 942.762, 6662.893 939.591, ...",1018
4,"POLYGON ((4819.988 2127.38, 4819.819 2124.872,...",3612
...,...,...
30356,"POLYGON ((15726.117 11652.8, 15726.048 11653.0...",34473
30357,"POLYGON ((16162.666 11640.066, 16162.644 11640...",34480
30357,"POLYGON ((16162.666 11640.066, 16162.644 11640...",34491
30430,"POLYGON ((16010.975 11698.536, 16010.99 11698....",34489


## Transfer the signals

In [10]:
intersection['index_st'] = intersection.index
intersection_st = intersection.set_index('index_st', drop = True)
intersection_sm = intersection.set_index('index_sm', drop = True)
intersection_st['cell_type'] = adata.obs.loc[intersection_st.index.to_numpy(), 'cell_type']

In [11]:
from tqdm import tqdm

In [12]:
kept_st_sm_pairs = []

for st_idx in tqdm(intersection_st.index.unique()):
    tmp_ct= intersection_st.loc[[st_idx],:]
    cell_segmentation = OpenFISH_gpd.loc[st_idx, 'geometry']

    sampling_area = 0
    kept_sm_idxs = []
    for sm_idx in tmp_ct['index_sm'].values:
        ablation_marker = MALDI_gpd.loc[sm_idx, 'geometry']

        main_sampling_area = ablation_marker.intersection(cell_segmentation).area

        if main_sampling_area > 0.1 * ablation_marker.area:
            # first inspect Sampling Specificity ratio
            total_sampling_area = 0
            tmp_sm = intersection_sm.loc[[sm_idx], :]
            
            for st_idx2 in tmp_sm['index_st'].values:
                cell_segmentation2 = OpenFISH_gpd.loc[st_idx2, 'geometry']
                total_sampling_area += ablation_marker.intersection(cell_segmentation2).area
    
            # Then Sampling Area
            if main_sampling_area / total_sampling_area >= 0.8 or len(intersection_st.loc[tmp_sm['index_st'].values, 'cell_type'].unique()) == 1: # Same cell type or only intersect with one cell
                
                sampling_area += main_sampling_area
                # record kept MALDI spots
                kept_sm_idxs.append(sm_idx)
        
    if sampling_area >= 0.3 * ablation_marker.area:     
        # Now the cells are qualified cells
        for sm_idx in kept_sm_idxs:
            kept_st_sm_pairs.append((st_idx, sm_idx))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20166/20166 [00:34<00:00, 581.73it/s]


In [13]:
intersection= intersection.set_index(['index_st', 'index_sm'], drop = False)
intersection = intersection.loc[kept_st_sm_pairs, :].copy()

In [14]:
def process_row(row):

    st_cell = OpenFISH_gpd.loc[row['index_st'],'geometry']
    sm_spot = MALDI_gpd.loc[row['index_sm'],'geometry']
    
    cell_area = st_cell.area
    ablation_area = sm_spot.area
    sampling_area = st_cell.intersection(sm_spot).area
        
    total_sampling_area = 0
    tmp_sm = intersection_sm.loc[[row['index_sm']], :]
    for st_idx2 in tmp_sm['index_st'].values:
        cell_segmentation2 = OpenFISH_gpd.loc[st_idx2, 'geometry']
        total_sampling_area += sm_spot.intersection(cell_segmentation2).area

    sampling_specificity = sampling_area / total_sampling_area

    if sampling_specificity< 0.8:
        sampling_specificity = 1.0

    return cell_area, ablation_area, sampling_area, sampling_specificity

In [15]:
intersection[['cell_areas', 'ablation_areas', 'sampling_areas', 'sampling_specificities']] = intersection.apply(process_row, axis=1, result_type='expand')

## Record data into AnnData

In [16]:
mdf = adata_m.to_df()
mdf = mdf.reset_index(drop = True)
mdf.index = mdf.index.astype(str)

In [17]:
new_frame_list = []
total_sampling_areas = []

for st_idx in tqdm(intersection['index_st'].unique()):

    tmp_inter = intersection.loc[[st_idx],['index_sm', 'ablation_areas', 'sampling_areas', 'sampling_specificities']].copy()
    sm_idxs = tmp_inter['index_sm'].values.reshape(-1,)

    total_sampling_area = (tmp_inter['sampling_areas'] * tmp_inter['sampling_specificities']).sum()
    
    multipliers = pd.Series((tmp_inter['sampling_areas'] / tmp_inter['ablation_areas'] * tmp_inter['sampling_specificities']).values, index=sm_idxs)

    tmp = mdf.loc[sm_idxs, :].copy()
    tmp = tmp.mul(multipliers, axis=0)
    
    new_frame_list.append(tmp.sum().to_frame(name = st_idx).T)
    total_sampling_areas.append(total_sampling_area)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14111/14111 [00:33<00:00, 421.30it/s]


In [18]:
final = pd.concat(new_frame_list)
adata_merged = sc.AnnData(final)
adata_merged.obs['total_sampling_area'] = total_sampling_areas
adata = adata[intersection['index_st'].unique().astype(str),:].copy()
adata_merged.obs['cell_type'] = adata.obs['cell_type']
adata_merged.obsm = adata.obsm

In [19]:
# Save these two aligned two modalities into two AnnData files
adata_merged.write_h5ad("demo_data/DHB_A3_transferMALDI.h5ad")
adata.write_h5ad("demo_data/DHB_A3_transferFISH.h5ad")